# Part 11 — A second disclosure regime: CFTC Commitments of Traders (Paper 1)
Paper 1 shows that Binance's class-level long/short ratios pin down memory rates that prices leave unrestricted. This notebook tests the same claim on **legally mandated** class-tagged disclosure in **regulated US futures**: the CFTC's weekly positions by trader class for Bitcoin, Ether, E-mini S&P 500, 10-year Treasury notes, euro, yen, crude oil and natural gas.

**Run C0, C1, C2** (C2 writes the pre-registration and downloads `P11_cot.json`; **e-mail it to yourself**), then **C3** (downloads the data, a few minutes), then **C4 once** (about 5–15 minutes). C4 downloads `p11_results.zip`; send it to Claude.

This notebook can run in its own Colab tab at the same time as Part 10.

In [ ]:
# ============================================================================
# CELL C0 — SETUP (self-contained; run first)
# Part 11 (Paper 1): a SECOND DISCLOSURE REGIME. The CFTC's weekly Commitments of Traders reports publish positions by
# trader class (leveraged funds, asset managers, dealers; managed money, producers, swap dealers) for regulated US futures.
# This is legally mandated class-tagged disclosure, in a different market and under a different regulator than Binance.
# ============================================================================
import os, sys, json, time, io, zipfile, hashlib, subprocess, warnings, glob
import urllib.request, urllib.error, urllib.parse
IN_COLAB = "google.colab" in sys.modules
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=RuntimeWarning)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/ceiling_suite"
else:
    BASE = os.path.abspath("ceiling_suite_local")
RESULTS, PREREG = os.path.join(BASE, "results"), os.path.join(BASE, "prereg")
CDIR = os.path.join(BASE, "data_cot")
for _d in (RESULTS, PREREG, CDIR):
    os.makedirs(_d, exist_ok=True)
SEED = 20261027

def _clean(o):
    if isinstance(o, dict):
        return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_clean(v) for v in o]
    if isinstance(o, np.ndarray):
        return _clean(o.tolist())
    if isinstance(o, (np.floating, float)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o

def record(test_id, passed, kind, metrics, criterion="", note=""):
    status = "REPORT" if passed is None else ("PASS" if passed else "FAIL")
    out = {"test_id": test_id, "status": status, "kind": kind, "criterion": criterion, "note": note,
           "metrics": _clean(metrics), "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "seed": SEED}
    json.dump(out, open(os.path.join(RESULTS, f"{test_id}.json"), "w"), indent=2)
    print(f"[{status}] {test_id} ({kind})" + (f" — {criterion}" if criterion else ""))
    return out

def _http(url, timeout=120, tries=4):
    last = None
    for i in range(tries):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=timeout) as r:
                return r.status, r.read()
        except urllib.error.HTTPError as e:
            if e.code in (400, 403, 404):
                return e.code, b""
            last = e
        except Exception as e:
            last = e
        time.sleep(2 * (i + 1))
    raise RuntimeError(f"network error {url}: {last}")

# ---- Markets (fixed before any data are downloaded) -------------------------------------------
# report: TFF = Traders in Financial Futures (futures only); DIS = Disaggregated (futures only)
# fast class = leveraged funds / managed money; slow class = asset managers / producer-merchants
MARKETS = {
    "BTC": {"code": "133741", "report": "TFF", "price": ("binance", "BTCUSDT")},
    "ETH": {"code": "146021", "report": "TFF", "price": ("binance", "ETHUSDT")},
    "ES":  {"code": "13874A", "report": "TFF", "price": ("yahoo", "^GSPC")},
    "TY":  {"code": "043602", "report": "TFF", "price": ("yahoo_yield", "^TNX")},
    "EUR": {"code": "099741", "report": "TFF", "price": ("yahoo", "EURUSD=X")},
    "JPY": {"code": "097741", "report": "TFF", "price": ("yahoo_inverse", "JPY=X")},
    "CL":  {"code": "067651", "report": "DIS", "price": ("yahoo", "CL=F")},
    "NG":  {"code": "023651", "report": "DIS", "price": ("yahoo", "NG=F")},
}
SOCRATA = {"TFF": "https://publicreporting.cftc.gov/resource/gpe5-46if.json",
           "DIS": "https://publicreporting.cftc.gov/resource/72hh-3qpy.json"}
CLASSES = {"TFF": {"fast": ("lev_money_positions_long", "lev_money_positions_short"),
                   "slow": ("asset_mgr_positions_long", "asset_mgr_positions_short"),
                   "dealer": ("dealer_positions_long_all", "dealer_positions_short_all")},
           "DIS": {"fast": ("m_money_positions_long_all", "m_money_positions_short_all"),
                   "slow": ("prod_merc_positions_long", "prod_merc_positions_short"),
                   "dealer": ("swap_positions_long_all", "swap__positions_short_all")}}
END_DATE = "2026-10-13"
code, _ = _http(SOCRATA["TFF"] + "?$limit=1", timeout=60)
print("CFTC public reporting API reachable:", code == 200, f"(HTTP {code})")
try:
    import yfinance as yf
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "yfinance"], check=True)
    import yfinance as yf
_t = yf.download("EURUSD=X", start="2024-01-01", end="2024-02-01", progress=False, auto_adjust=False)
print("Yahoo Finance reachable:", len(_t) > 10, f"({len(_t)} rows)")
code2, _ = _http("https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1d/BTCUSDT-1d-2024-01.zip", timeout=60)
print("Binance archive reachable:", code2 == 200, f"(HTTP {code2})")
assert code == 200 and code2 == 200 and len(_t) > 10, "a data source is not reachable from this runtime; tell Claude."

In [ ]:
# ============================================================================
# CELL C1 — FUNCTIONS
# ============================================================================
def get_cot(mkt):
    m = MARKETS[mkt]; fp = os.path.join(CDIR, f"cot_{mkt}.csv")
    if os.path.exists(fp):
        return pd.read_csv(fp, parse_dates=["date"])
    q = {"cftc_contract_market_code": m["code"], "$limit": "50000", "$order": "report_date_as_yyyy_mm_dd"}
    code, raw = _http(SOCRATA[m["report"]] + "?" + urllib.parse.urlencode(q), timeout=300)
    assert code == 200, f"COT download failed for {mkt}: HTTP {code}"
    df = pd.DataFrame(json.loads(raw))
    cols = CLASSES[m["report"]]
    out = pd.DataFrame({"date": pd.to_datetime(df["report_date_as_yyyy_mm_dd"]).dt.tz_localize(None).dt.normalize(),
                        "oi": pd.to_numeric(df["open_interest_all"], errors="coerce"),
                        "market_name": df["market_and_exchange_names"]})
    for c, (lc, sc) in cols.items():
        out[f"{c}_long"] = pd.to_numeric(df[lc], errors="coerce"); out[f"{c}_short"] = pd.to_numeric(df[sc], errors="coerce")
    out = out.drop_duplicates("date").sort_values("date")
    out = out[out["date"] <= END_DATE].reset_index(drop=True)
    out.to_csv(fp, index=False)
    return out

def get_binance_daily(sym):
    fp = os.path.join(CDIR, f"px_{sym}.csv")
    if os.path.exists(fp):
        return pd.read_csv(fp, parse_dates=["date"]).set_index("date")["px"]
    rows = []
    for mth in pd.period_range("2017-08", END_DATE[:7], freq="M"):
        url = f"https://data.binance.vision/data/spot/monthly/klines/{sym}/1d/{sym}-1d-{mth}.zip"
        code, raw = _http(url, timeout=120)
        if code != 200 or not raw:
            continue
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            k = pd.read_csv(z.open(z.namelist()[0]), header=None)
        t = pd.to_numeric(k[0]); t = np.where(t > 1e14, t / 1000, t)          # microseconds from 2025 on
        rows.append(pd.DataFrame({"date": pd.to_datetime(t, unit="ms").normalize(), "px": k[4].astype(float)}))
    s = pd.concat(rows).drop_duplicates("date").sort_values("date")
    s.to_csv(fp, index=False)
    return s.set_index("date")["px"]

def get_yahoo(ticker):
    fp = os.path.join(CDIR, "yahoo_" + ticker.replace("^", "").replace("=", "_") + ".csv")
    if not os.path.exists(fp):
        d = yf.download(ticker, start="2005-01-01", end=END_DATE, progress=False, auto_adjust=False)
        assert len(d) > 100, f"Yahoo {ticker}: no data"
        c = d["Close"]
        c = c.iloc[:, 0] if isinstance(c, pd.DataFrame) else c          # newer yfinance returns a one-column frame
        pd.DataFrame({"date": pd.to_datetime(c.index).tz_localize(None).normalize(), "v": c.to_numpy(float)}).to_csv(fp, index=False)
    d = pd.read_csv(fp, parse_dates=["date"]).dropna()
    d = d[d["v"] > 0].drop_duplicates("date")
    return d.set_index("date")["v"]

def weekly_panel(mkt):
    """COT report dates (Tuesdays) with class net positions as shares of open interest and the weekly return
    from the last price on or before each report date."""
    cot = get_cot(mkt); kind, sid = MARKETS[mkt]["price"]
    if kind == "binance":
        px = np.log(get_binance_daily(sid)); px.name = "lp"
    else:
        v = get_yahoo(sid)
        px = {"yahoo": np.log(v), "yahoo_inverse": -np.log(v), "yahoo_yield": -v}[kind]   # yield (^TNX, percent): return proxy = -change
    px = px.sort_index()
    lp = px.reindex(cot["date"], method="ffill").to_numpy()
    stale = (cot["date"] - px.index[np.clip(np.searchsorted(px.index, cot["date"], side="right") - 1, 0, len(px) - 1)]).dt.days.to_numpy()
    D = pd.DataFrame({"date": cot["date"], "lp": lp, "stale_days": stale})
    for c in ("fast", "slow", "dealer"):
        D[f"n_{c}"] = (cot[f"{c}_long"] - cot[f"{c}_short"]) / cot["oi"]
        D[f"n_{c}_lag"] = D[f"n_{c}"].shift(1)
    D["ret"] = D["lp"].diff()
    ok = (D["date"].diff().dt.days == 7) & (D["stale_days"] <= 4) & (D["stale_days"].shift(1) <= 4)
    D = D[ok].dropna().reset_index(drop=True)                 # consecutive weekly reports with recent prices
    return D

def tagged_fit(D, c):
    X = np.column_stack([np.ones(len(D)), D[f"n_{c}_lag"], D["ret"]]); y = D[f"n_{c}"].to_numpy()
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    return b[1], b[2]                                          # memory rate r_c, response to returns w_c

# price-only alternative: weekly returns on 12 own lags, then a two-exponential kernel phi_j = sum_i b_i r_i^(j-1)
LAGS = 12
RGRID = np.round(np.arange(0.0, 0.99 + 1e-9, 0.02), 2)
PAIRS = np.array([(r1, r2) for i, r1 in enumerate(RGRID) for r2 in RGRID[i + 1:]])
_J = np.arange(LAGS)
_RES = []
for r1, r2 in PAIRS:
    Xp = np.column_stack([r1 ** _J, r2 ** _J]); Hp = Xp @ np.linalg.pinv(Xp)
    _RES.append(np.eye(LAGS) - Hp)
_RES = np.array(_RES)                                           # (pairs, 12, 12)
def ar_coefs(ret):
    y = ret[LAGS:]; X = np.column_stack([np.ones(len(y))] + [ret[LAGS - j:-j] for j in range(1, LAGS + 1)])
    return np.linalg.lstsq(X, y, rcond=None)[0][1:]
def two_exp(phi):
    sse = np.einsum("i,pij,j->p", phi, _RES, phi)
    return PAIRS[int(np.argmin(sse))]

def block_index(T, block, R):
    nb = int(np.ceil(T / block)); starts = R.integers(0, T - block + 1, nb)
    return np.concatenate([np.arange(s, s + block) for s in starts])[:T]

def analyse(mkt, B=2000, block=13):
    D = weekly_panel(mkt); T = len(D); R = np.random.default_rng(SEED + sum(map(ord, mkt)))
    est = {c: tagged_fit(D, c) for c in ("fast", "slow", "dealer")}
    ret = D["ret"].to_numpy(); po = two_exp(ar_coefs(ret))
    bt = {c: [] for c in ("fast", "slow", "dealer")}; bpo = []
    for b in range(B):
        idx = block_index(T, block, R); Db = D.iloc[idx].reset_index(drop=True)
        for c in bt:
            bt[c].append(tagged_fit(Db, c))
        rb = ret[block_index(len(ret), block, R)]           # price-only: resample return blocks, refit
        bpo.append(two_exp(ar_coefs(rb)))
    q = lambda v: [float(np.quantile(v, 0.025)), float(np.quantile(v, 0.975))]
    out = {"weeks": T, "first": str(D["date"].iloc[0].date()), "last": str(D["date"].iloc[-1].date())}
    for c in bt:
        a = np.array(bt[c])
        out[c] = {"r": float(est[c][0]), "r_ci": q(a[:, 0]), "w": float(est[c][1]), "w_ci": q(a[:, 1]),
                  "halflife_weeks": float(np.log(0.5) / np.log(est[c][0])) if 0 < est[c][0] < 1 else None}
    dfs = np.array(bt["fast"])[:, 0] - np.array(bt["slow"])[:, 0]
    out["fast_minus_slow_r"] = {"est": float(est["fast"][0] - est["slow"][0]), "ci": q(dfs)}
    bpo = np.array(bpo)
    out["price_only"] = {"r1": float(po[0]), "r2": float(po[1]), "r1_ci": q(bpo[:, 0]), "r2_ci": q(bpo[:, 1])}
    out["tagged_width_fast"] = out["fast"]["r_ci"][1] - out["fast"]["r_ci"][0]
    out["price_only_width"] = float(np.mean([out["price_only"]["r1_ci"][1] - out["price_only"]["r1_ci"][0],
                                             out["price_only"]["r2_ci"][1] - out["price_only"]["r2_ci"][0]]))
    out["width_ratio"] = out["price_only_width"] / out["tagged_width_fast"] if out["tagged_width_fast"] > 0 else None
    return out, D

## Pre-registration (before any data are downloaded)

In [ ]:
# ============================================================================
# CELL C2 — PRE-REGISTRATION (before any COT or price data are downloaded). E-mail P11_cot.json to yourself.
# ============================================================================
PREREG_P11 = {
    "question": "Does legally mandated class-tagged disclosure (CFTC Commitments of Traders) pin down trader-class memory "
                "rates that prices alone leave unrestricted, and do the classes behave as the cohort model predicts?",
    "markets": {k: {"code": v["code"], "report": v["report"], "price": list(v["price"])} for k, v in MARKETS.items()},
    "classes": {"TFF": "fast = leveraged funds, slow = asset managers, dealer = dealers",
                "DIS": "fast = managed money, slow = producer/merchant, dealer = swap dealers"},
    "data": "CFTC public reporting API (futures-only reports), all weekly reports through " + END_DATE +
            "; prices: Binance spot daily close (BTC, ETH); Yahoo Finance daily close for the others: S&P 500 index, 10-year yield "
            "(enters as minus the change), EUR/USD, USD/JPY (inverted), and front-month crude-oil and natural-gas futures. "
            "Class position n = (long - short) / open interest; weekly return from the last price on or before the report date "
            "(at most 4 days old); only consecutive weekly reports.",
    "inclusion": "a market enters if it has at least 150 usable weeks",
    "tagged_model": "n_t = alpha + r * n_{t-1} + w * ret_t + e_t by OLS, for each class",
    "price_only_model": "weekly returns on 12 own lags (OLS), then a two-exponential kernel fitted on a 0.02 grid of rates in [0, 0.99]",
    "inference": "moving-block bootstrap, 13-week blocks, 2,000 draws; 95% percentile intervals",
    "H1_primary": "disclosure precision: the fast-class rate interval is narrower than 0.25 AND both price-only rate intervals are "
                  "wider than 0.25; success if this holds in at least 75% of included markets",
    "H2": "trend following: the fast-class response to returns w is positive with interval above 0, in at least 75% of markets",
    "H3": "classes differ: the fast class has a shorter memory than the slow class (interval of r_fast - r_slow below 0), "
          "in at least half of the markets",
    "runs": "C4 is run once; every run is counted in prereg/test_set_unlocks_p11.json",
    "relation": "Paper 1 Table tab:disc (Binance long/short ratios, Part 6) is the first disclosure regime; this is the second.",
}
spec11 = json.dumps(PREREG_P11, indent=2, sort_keys=True); P11_HASH = hashlib.sha256(spec11.encode()).hexdigest()
p11_path = os.path.join(PREREG, "P11_cot.json")
if not os.path.exists(p11_path):
    open(p11_path, "w").write(spec11); print("pre-registration written:", p11_path)
    if IN_COLAB:
        from google.colab import files
        files.download(p11_path)
elif hashlib.sha256(open(p11_path).read().encode()).hexdigest() != P11_HASH:
    raise RuntimeError("The spec differs from the registered file. Do not edit the spec after registering it.")
print("spec SHA-256:", P11_HASH)

## Data

In [ ]:
# ============================================================================
# CELL C3 — DOWNLOAD AND CHECK DATA (no estimate is computed here)
# ============================================================================
assert hashlib.sha256(open(p11_path).read().encode()).hexdigest() == P11_HASH, "run C2 first"
COVER = {}
for mkt in MARKETS:
    try:
        cot = get_cot(mkt); D = weekly_panel(mkt)
        COVER[mkt] = {"market_name": str(cot["market_name"].iloc[-1]), "reports": len(cot),
                      "first_report": str(cot["date"].iloc[0].date()), "usable_weeks": len(D)}
    except Exception as e:
        COVER[mkt] = {"error": repr(e)[:200], "usable_weeks": 0}
    print(mkt, COVER[mkt])
INCLUDED = [m for m in MARKETS if COVER[m].get("usable_weeks", 0) >= 150]
print("included markets:", INCLUDED)
record("C3_cot_data", None, "audit", {"coverage": COVER, "included": INCLUDED})

## The test: run ONCE

In [ ]:
# ============================================================================
# CELL C4 — THE TEST (RUN ONCE; every run is counted). About 5-15 minutes. Downloads p11_results.zip.
# ============================================================================
assert hashlib.sha256(open(p11_path).read().encode()).hexdigest() == P11_HASH, "spec changed after registration"
pth11 = os.path.join(PREREG, "test_set_unlocks_p11.json")
n11 = (json.load(open(pth11))["n"] if os.path.exists(pth11) else 0) + 1
json.dump({"n": n11, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "spec_sha256": P11_HASH}, open(pth11, "w"))
RES = {}
for mkt in INCLUDED:
    t0 = time.time(); RES[mkt], _ = analyse(mkt)
    r = RES[mkt]
    print(f"{mkt}: {r['weeks']} wks | fast r={r['fast']['r']:.3f} {np.round(r['fast']['r_ci'],3)} w={r['fast']['w']:.3f} "
          f"{np.round(r['fast']['w_ci'],3)} | slow r={r['slow']['r']:.3f} | fast-slow {np.round(r['fast_minus_slow_r']['ci'],3)} | "
          f"price-only widths {r['price_only_width']:.2f} vs tagged {r['tagged_width_fast']:.3f} ({time.time()-t0:.0f}s)")
h1 = {m: bool(RES[m]["tagged_width_fast"] < 0.25 and
              all(RES[m]["price_only"][k][1] - RES[m]["price_only"][k][0] > 0.25 for k in ("r1_ci", "r2_ci"))) for m in RES}
h2 = {m: bool(RES[m]["fast"]["w_ci"][0] > 0) for m in RES}
h3 = {m: bool(RES[m]["fast_minus_slow_r"]["ci"][1] < 0) for m in RES}
n = len(RES)
ok1, ok2, ok3 = sum(h1.values()) >= 0.75 * n, sum(h2.values()) >= 0.75 * n, sum(h3.values()) >= 0.5 * n
print(f"\n=== H1 (primary, disclosure precision): {sum(h1.values())}/{n} ->", "PASS" if ok1 else "FAIL")
print(f"=== H2 (fast class trend-follows):      {sum(h2.values())}/{n} ->", "PASS" if ok2 else "FAIL")
print(f"=== H3 (fast class has shorter memory): {sum(h3.values())}/{n} ->", "PASS" if ok3 else "FAIL")
record("C4_cot_disclosure", ok1, "real-data", {"by_market": RES, "H1": h1, "H2": h2, "H3": h3, "H2_pass": ok2, "H3_pass": ok3,
       "prereg_sha256": P11_HASH, "test_set_unlocks": n11}, criterion=PREREG_P11["H1_primary"])
zp = os.path.join(BASE, "p11_results.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for fp in [os.path.join(RESULTS, "C3_cot_data.json"), os.path.join(RESULTS, "C4_cot_disclosure.json"), p11_path, pth11]:
        if os.path.exists(fp):
            z.write(fp, os.path.relpath(fp, BASE))
print("zipped ->", zp)
if IN_COLAB:
    from google.colab import files
    files.download(zp)

If C0 says a source is unreachable, or C3 includes fewer than 6 markets, tell Claude before running C4.